# PDF データ抽出 (Google Colab)

権利面をかなりシンプルにした構成です。本文・表・メタデータ・添付ファイルを抽出し、必要なら OCR も追加できます。

In [ ]:
# 本文・表・メタデータ抽出だけならこれでOK
!pip -q install pdfplumber pypdf pandas openpyxl

# スキャンPDFにも対応したい場合は、こちらも追加で実行
# !pip -q install pypdfium2 pillow pytesseract
# !apt-get -qq update
# !apt-get -qq install -y tesseract-ocr tesseract-ocr-jpn


In [ ]:
from google.colab import files
uploaded = files.upload()

pdf_files = [name for name in uploaded.keys() if name.lower().endswith('.pdf')]
if not pdf_files:
    raise ValueError('PDFファイルをアップロードしてください。')

PDF_PATH = pdf_files[0]
OUT_DIR = 'pdf_extract_result'
USE_OCR = False
OCR_LANG = 'jpn+eng'
MIN_TEXT_CHARS_FOR_DIGITAL_PDF = 20
TABLE_SETTINGS = {
    'vertical_strategy': 'lines',
    'horizontal_strategy': 'lines',
    'snap_tolerance': 3,
    'join_tolerance': 3,
    'intersection_tolerance': 3,
}
print(f'対象PDF: {PDF_PATH}')
print(f'OCR使用: {USE_OCR}')


In [ ]:
from pathlib import Path
import json
import re
import shutil
import pandas as pd
import pdfplumber
from pypdf import PdfReader

def safe_name(name: str) -> str:
    name = name or "unnamed"
    name = re.sub(r'[\\/:*?"<>|\r\n]+', "_", name)
    return name[:180]

def to_jsonable(value):
    if value is None:
        return None
    if isinstance(value, (str, int, float, bool)):
        return value
    try:
        return value.isoformat()
    except Exception:
        return str(value)

def write_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def extract_pdf(
    pdf_path: str,
    out_dir: str,
    use_ocr: bool = False,
    ocr_lang: str = "jpn+eng",
    ocr_scale: float = 2.0,
    min_text_chars_for_digital_pdf: int = 20,
    table_settings: dict | None = None,
):
    pdf_path = Path(pdf_path)
    out_dir = Path(out_dir)

    if out_dir.exists():
        shutil.rmtree(out_dir)

    (out_dir / "pages").mkdir(parents=True, exist_ok=True)
    (out_dir / "tables").mkdir(exist_ok=True)
    (out_dir / "attachments").mkdir(exist_ok=True)

    reader = PdfReader(str(pdf_path))
    metadata = {}
    if reader.metadata:
        for k, v in reader.metadata.items():
            metadata[str(k)] = to_jsonable(v)

    attachments_info = []
    try:
        attachments = reader.attachments
    except Exception:
        attachments = {}

    if attachments:
        for name, content_list in attachments.items():
            safe = safe_name(name)
            for i, content in enumerate(content_list, start=1):
                fname = f"{i:02d}_{safe}"
                out = out_dir / "attachments" / fname
                with open(out, "wb") as f:
                    f.write(content)
                attachments_info.append({
                    "name": name,
                    "saved_as": fname,
                    "bytes": len(content),
                })

    all_pages = []
    all_words = []
    table_registry = []
    ocr_rows = []

    if use_ocr:
        try:
            import pypdfium2 as pdfium
            import pytesseract
            from pytesseract import Output
        except ImportError as e:
            raise ImportError(
                "OCR を使うには pypdfium2 / pytesseract / Pillow / Tesseract をインストールしてください。"
            ) from e

        pdfium_doc = pdfium.PdfDocument(str(pdf_path))
    else:
        pdfium_doc = None

    with pdfplumber.open(str(pdf_path)) as pdf:
        for page_idx, page in enumerate(pdf.pages, start=1):
            text = page.extract_text(x_tolerance=2, y_tolerance=3) or ""
            text_source = "pdf_text"

            # 単語＋座標を保存
            words = page.extract_words(extra_attrs=["fontname", "size"]) or []
            for w in words:
                w2 = {k: to_jsonable(v) for k, v in w.items()}
                w2["page_number"] = page_idx
                all_words.append(w2)

            # デジタル文字がほぼ無いページだけ OCR
            if use_ocr and len(text.strip()) < min_text_chars_for_digital_pdf:
                bitmap = pdfium_doc[page_idx - 1].render(scale=ocr_scale)
                image = bitmap.to_pil()

                ocr_text = pytesseract.image_to_string(
                    image,
                    lang=ocr_lang,
                    config="--psm 6",
                )

                if len(ocr_text.strip()) > len(text.strip()):
                    text = ocr_text
                    text_source = "ocr"

                ocr_df = pytesseract.image_to_data(
                    image,
                    lang=ocr_lang,
                    config="--psm 6",
                    output_type=Output.DATAFRAME,
                )

                if not ocr_df.empty:
                    ocr_df = ocr_df.dropna(subset=["text"]).copy()
                    ocr_df["text"] = ocr_df["text"].astype(str)
                    ocr_df = ocr_df[ocr_df["text"].str.strip() != ""]
                    if not ocr_df.empty:
                        ocr_df["page_number"] = page_idx
                        ocr_rows.append(ocr_df)

            # ページ本文保存
            page_payload = {
                "page_number": page_idx,
                "width": page.width,
                "height": page.height,
                "text_source": text_source,
                "text": text,
            }
            all_pages.append(page_payload)

            with open(out_dir / "pages" / f"page_{page_idx:04d}.txt", "w", encoding="utf-8") as f:
                f.write(text)

            # 表抽出
            tables = page.extract_tables(table_settings=table_settings or {}) or []
            for table_idx, table in enumerate(tables, start=1):
                df = pd.DataFrame(table)
                csv_name = f"page_{page_idx:04d}_table_{table_idx:02d}.csv"
                df.to_csv(
                    out_dir / "tables" / csv_name,
                    index=False,
                    header=False,
                    encoding="utf-8-sig",
                )
                table_registry.append({
                    "page_number": page_idx,
                    "table_number": table_idx,
                    "rows": len(df),
                    "cols": len(df.columns),
                    "csv_file": csv_name,
                })

    # JSON系
    write_json(metadata, out_dir / "metadata.json")
    write_json(all_pages, out_dir / "pages_text.json")
    write_json(table_registry, out_dir / "tables_index.json")
    write_json(attachments_info, out_dir / "attachments_index.json")

    # 全文
    with open(out_dir / "all_text.txt", "w", encoding="utf-8") as f:
        for p in all_pages:
            f.write(f"\n===== PAGE {p['page_number']} ({p['text_source']}) =====\n")
            f.write(p["text"] or "")
            f.write("\n")

    # 単語座標CSV
    if all_words:
        pd.DataFrame(all_words).to_csv(
            out_dir / "words.csv",
            index=False,
            encoding="utf-8-sig",
        )

    # OCR座標CSV
    if ocr_rows:
        pd.concat(ocr_rows, ignore_index=True).to_csv(
            out_dir / "ocr_words.csv",
            index=False,
            encoding="utf-8-sig",
        )

    # 表を1つのExcelにまとめる
    if table_registry:
        with pd.ExcelWriter(out_dir / "tables.xlsx", engine="openpyxl") as writer:
            for item in table_registry:
                csv_path = out_dir / "tables" / item["csv_file"]
                df = pd.read_csv(csv_path, header=None)
                sheet_name = f"p{item['page_number']:03d}_t{item['table_number']:02d}"
                df.to_excel(writer, sheet_name=sheet_name, index=False, header=False)

    summary = {
        "source_pdf": pdf_path.name,
        "page_count": len(all_pages),
        "tables_found": len(table_registry),
        "attachments_found": len(attachments_info),
        "ocr_enabled": use_ocr,
        "ocr_pages_used": sum(1 for p in all_pages if p["text_source"] == "ocr"),
        "output_examples": [
            "metadata.json",
            "pages_text.json",
            "all_text.txt",
            "pages/page_0001.txt",
            "words.csv",
            "tables_index.json",
            "tables/page_0001_table_01.csv",
            "tables.xlsx",
            "attachments/*",
            "ocr_words.csv",
        ],
    }
    write_json(summary, out_dir / "summary.json")
    return summary



In [ ]:
summary = extract_pdf(
    pdf_path=PDF_PATH,
    out_dir=OUT_DIR,
    use_ocr=USE_OCR,
    ocr_lang=OCR_LANG,
    min_text_chars_for_digital_pdf=MIN_TEXT_CHARS_FOR_DIGITAL_PDF,
    table_settings=TABLE_SETTINGS,
)

print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive(OUT_DIR, 'zip', root_dir='.', base_dir=OUT_DIR)
files.download(zip_path)
